# Aegis Wave - Load-then-Train Smoke Test
## Goal: Avoid deadlocks by loading all data into RAM before starting training.

This test isolates the **AI training** from the **Disk I/O**.

In [6]:
import os
import h5py
import numpy as np
import pandas as pd
import json
import tensorflow as tf
from tensorflow import keras
import time

# EXTREME SPEED CONSTANTS
DATA_DIR = "data/"
SUB_COUNT = 16   # Take every 4th subcarrier
TIME_STEPS = 100 # Crop to first 100 packets
SAMPLE_LIMIT = 100 # Total samples for this test
LABEL_MAP = {"Fall": 0, "Nonfall": 1}

# Force CPU to be absolutely sure
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
tf.config.set_visible_devices([], "GPU")

In [7]:
print("Step 1: Loading HDF5 data into RAM...")
X_data = []
y_data = []

metadata_df = pd.read_csv(os.path.join(DATA_DIR, "metadata/sample_metadata.csv"))
with open(os.path.join(DATA_DIR, "splits/train_id.json"), "r") as f:
    ids = json.load(f)[:SAMPLE_LIMIT]

for i, sample_id in enumerate(ids):
    row = metadata_df[metadata_df["id"] == sample_id]
    if row.empty: continue
    
    file_path = os.path.join(DATA_DIR, row.iloc[0]["file_path"].lstrip("./"))
    try:
        with h5py.File(file_path, "r") as f:
            data = f["CSI_amps"][:].squeeze()
            if data.shape[1] < TIME_STEPS: continue
            data = data[:64:4, :TIME_STEPS].T # Transpose to (100, 16)
            X_data.append(data.astype("float32"))
            y_data.append(LABEL_MAP[row.iloc[0]["label"]])
        if (i+1) % 20 == 0: print(f"  Loaded {i+1}/{SAMPLE_LIMIT} files")
    except Exception as e:
        continue

X_train = np.array(X_data)
y_train = np.array(y_data)
print(f"SUCCESS: Loaded {len(X_train)} samples into RAM.")
print(f"Data Shape: {X_train.shape}")

Step 1: Loading HDF5 data into RAM...
  Loaded 20/100 files
  Loaded 40/100 files
  Loaded 60/100 files
  Loaded 80/100 files
  Loaded 100/100 files
SUCCESS: Loaded 99 samples into RAM.
Data Shape: (99, 100, 16)


In [ ]:
print("Step 2: Starting Model Fit on pre-loaded memory...")
model = keras.Sequential([
    keras.layers.Input(shape=(TIME_STEPS, SUB_COUNT)),
    keras.layers.Conv1D(16, 3, activation="relu"),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(2, activation="softmax")
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
start = time.time()
model.fit(X_train, y_train, epochs=5, batch_size=16, verbose=1)
print(f"✅ SMOKE TEST COMPLETED! Training time: {time.time() - start:.2f}s")

Step 2: Starting Model Fit on pre-loaded memory...
Epoch 1/5
